# 01 — Data Collection
### Kenya Clean Cooking Access Gap Analysis

This notebook guides you through downloading and inspecting all source datasets needed for the analysis. Each section covers one data source, with instructions for manual downloads where APIs aren't available.

**Run order:** This is notebook 1 of 5. Run before all others.

---
## 0. Setup

In [1]:
#%pip install wbdata pandas geopandas requests pyreadstat
import os
import requests
import pandas as pd
import geopandas as gpd
import wbdata
from pathlib import Path
from datetime import datetime
import pyreadstat

# Project paths
ROOT = Path('..').resolve()
RAW = ROOT / 'data' / 'raw'
SHAPEFILES = ROOT / 'data' / 'shapefiles'

RAW.mkdir(parents=True, exist_ok=True)
SHAPEFILES.mkdir(parents=True, exist_ok=True)

print(f'Root: {ROOT}')
print(f'Raw data: {RAW}')

Root: C:\Users\eric.njiraini\Documents\My Projects at a Go\kenya-clean-cooking-gap
Raw data: C:\Users\eric.njiraini\Documents\My Projects at a Go\kenya-clean-cooking-gap\data\raw


---
## 1. World Bank SDG-7 Time Series (API)
Access to clean fuels and technologies for cooking (% of population) — national level, Kenya.

In [3]:
# World Bank indicator: EG.CFT.ACCS.ZS = Access to clean fuels for cooking (%)
indicators = {
    'EG.CFT.ACCS.ZS': 'clean_cooking_access_pct',
    'SP.POP.TOTL':     'population_total',
    'NY.GDP.PCAP.CD':  'gdp_per_capita_usd',
    'EG.ELC.ACCS.ZS':  'electricity_access_pct',
}

df_wb = wbdata.get_dataframe(
    indicators,
    country='KE',
)
df_wb = df_wb.reset_index()
df_wb = df_wb.rename(columns={'date': 'year'})
df_wb['year'] = df_wb['year'].astype(int)
df_wb = df_wb.sort_values('year')

print(f'Shape: {df_wb.shape}')
print(f'Years: {df_wb.year.min()} — {df_wb.year.max()}')
df_wb.tail(10)

Shape: (66, 5)
Years: 1960 — 2025


,year,clean_cooking_access_pct,population_total,gdp_per_capita_usd,electricity_access_pct
9,2016,14.20,48139687.0,1554.126103,53.1
8,2017,16.10,49197766.0,1667.484472,55.8
7,2018,18.00,50207107.0,1836.452755,61.2
6,2019,20.60,51202827.0,1960.408089,69.7
5,2020,23.05,52217334.0,1927.664590,71.5
4,2021,25.80,53219166.0,2061.356221,76.5
3,2022,28.50,54252461.0,2109.562885,76.0
2,2023,31.60,55339003.0,1942.588028,76.2
1,2024,NaN,56432944.0,2132.434521,NaN
0,2025,NaN,NaN,NaN,NaN


In [4]:
# Save
df_wb.to_csv(RAW / 'worldbank_kenya_sdg7_timeseries.csv', index=False)
print('Saved worldbank_kenya_sdg7_timeseries.csv')

Saved worldbank_kenya_sdg7_timeseries.csv


---
## 2. Kenya County Shapefiles (GADM)

**Manual step required:**
1. Go to https://gadm.org/download_country.html
2. Select **Kenya** → **Shapefile**
3. Download `gadm41_KEN_shp.zip`
4. Unzip into `data/shapefiles/`
5. The file you need is `gadm41_KEN_1.shp` (level 1 = counties)

**Alternative (HDX):** https://data.humdata.org/dataset/kenya-counties
Download the Kenya counties GeoJSON — already has county names matched to official list.

In [5]:
# Load and inspect once downloaded
shapefile_path = SHAPEFILES / 'gadm41_KEN_1.shp'

if shapefile_path.exists():
    gdf = gpd.read_file(shapefile_path)
    print(f'CRS: {gdf.crs}')
    print(f'Counties found: {len(gdf)}')
    print(gdf[['NAME_1', 'GID_1']].head(10))
else:
    print('⚠️  Shapefile not yet downloaded. Follow the manual step above.')

CRS: EPSG:4326
Counties found: 47
            NAME_1     GID_1
0          Baringo   KEN.1_1
1            Bomet   KEN.2_1
2          Bungoma   KEN.3_1
3            Busia   KEN.4_1
4  Elgeyo-Marakwet   KEN.5_1
5             Embu   KEN.6_1
6          Garissa   KEN.7_1
7         Homa Bay   KEN.8_1
8           Isiolo   KEN.9_1
9          Kajiado  KEN.10_1


---
## 3. Kenya DHS 2022 — Household Fuel Data

**Manual step required** (one-time registration):
1. Register at https://dhsprogram.com/data/new-user-registration.cfm (free)
2. Request the **Kenya 2022 DHS** — Household Recode dataset (HR file)
3. Download the Stata (.dta) or CSV version
4. Save as `data/raw/KEHR8AFL.csv` (or .dta)

**Key variable:** `HV226` — Type of cooking fuel

| Code | Fuel Type |
|------|-----------|
| 1 | Electricity |
| 2 | LPG/natural gas |
| 3 | Biogas |
| 4 | Kerosene |
| 5 | Coal/lignite |
| 6 | Charcoal |
| 7 | Wood |
| 8 | Straw/shrubs/grass |
| 9 | Agricultural crop |
| 10 | Animal dung |
| 95 | No food cooked in HH |
| 96 | Other |

WHO clean = codes 1, 2, 3 (electricity, LPG, biogas)

In [6]:
# Define clean fuel codes (WHO standard)
CLEAN_FUEL_CODES = [1, 2, 3]  # Electricity, LPG/natural gas, Biogas

# Load DHS data once available
dhs_path = RAW / 'KEHR8CFL.dta'

if dhs_path.exists():
    df_dhs, meta = pyreadstat.read_dta(dhs_path)
    df_dhs.columns = df_dhs.columns.str.upper()
    # df_dhs = pd.read_stata(dhs_path)
    
    # Key columns to retain
    cols = [
        'HV001',   # Cluster number
        'HV002',   # Household number  
        'HV024',   # Region/county
        'HV226',   # Type of cooking fuel
        'HV271',   # Wealth index factor score
        'HV270',   # Wealth index quintile
        'HV009',   # Number of household members
        'HV219',   # Sex of household head (1=male, 2=female)
        'HV025',   # Type of place of residence (1=urban, 2=rural)
        'HV005',   # Household sample weight
    ]
    
    available = [c for c in cols if c in df_dhs.columns]
    df_dhs = df_dhs[available].copy()
    
    # Create binary clean cooking indicator
    df_dhs['clean_cooking'] = df_dhs['HV226'].isin(CLEAN_FUEL_CODES).astype(int)
    
    print(f'DHS records loaded: {len(df_dhs):,}')
    print(f'\nFuel type distribution:')
    print(df_dhs['HV226'].value_counts().head(10))
    print(f'\nOverall clean cooking rate: {df_dhs["clean_cooking"].mean():.1%}')
else:
    print('⚠️  DHS file not yet downloaded. Follow manual step above.')
    print('\nUsing placeholder county data for pipeline development...')
    
    # Placeholder: published county-level data from KNCTS 2024 strategy
    # Source: Kenya National Cooking Transition Strategy 2024-2028 (Table 3)
    df_county_placeholder = pd.DataFrame({
        'county': [
            'Nairobi', 'Mombasa', 'Kisumu', 'Nakuru', 'Uasin Gishu',
            'Kiambu', 'Machakos', 'Turkana', 'Mandera', 'Wajir',
            'Garissa', 'Marsabit', 'Isiolo', 'Samburu', 'West Pokot'
        ],
        'clean_cooking_pct': [
            72, 58, 41, 38, 35,
            45, 28, 4, 3, 5,
            8, 6, 12, 7, 6
        ],
        'population_2019': [
            4397073, 1208333, 1155574, 2162202, 1163186,
            2417735, 1421932, 926976, 867457, 781263,
            841353, 459785, 268002, 310327, 621241
        ]
    })
    print(df_county_placeholder)

DHS records loaded: 37,911

Fuel type distribution:
HV226
8     22129
2      7769
7      5867
5       865
95      632
3       311
9        85
1        78
4        73
12       37
Name: count, dtype: int64

Overall clean cooking rate: 21.5%


---
## 4. Facebook Relative Wealth Index (HDX)

**Manual download:**
1. Go to https://data.humdata.org/dataset/relative-wealth-index
2. Find **Kenya** CSV
3. Save as `data/raw/ken_relative_wealth_index.csv`

This gives a sub-national wealth score per ~2.4km grid cell — we'll aggregate to county level.

In [7]:
rwi_path = RAW / 'ken_relative_wealth_index.csv'

if rwi_path.exists():
    df_rwi = pd.read_csv(rwi_path)
    print(f'RWI records: {len(df_rwi):,}')
    print(df_rwi.head())
    print(f'\nRWI range: {df_rwi["rwi"].min():.3f} to {df_rwi["rwi"].max():.3f}')
else:
    print('⚠️  RWI file not yet downloaded. Follow manual step above.')

RWI records: 29,483
   latitude  longitude    rwi  error
0  0.318602  34.749756  0.707  0.522
1  1.131518  36.573486 -0.344  0.425
2 -0.714093  36.815185 -0.118  0.460
3  0.582265  39.364014 -0.698  0.364
4  1.241358  35.870361 -0.584  0.351

RWI range: -1.625 to 1.753


---
## 5. Data Inventory Check

In [8]:
datasets = {
    'World Bank SDG-7 time series':  RAW / 'worldbank_kenya_sdg7_timeseries.csv',
    'Kenya county shapefile':        SHAPEFILES / 'gadm41_KEN_1.shp',
    'Kenya DHS 2022 household data': RAW / 'KEHR8AFL.csv',
    'Facebook Relative Wealth Index': RAW / 'ken_relative_wealth_index.csv',
}

print('📦 Data Inventory')
print('=' * 55)
for name, path in datasets.items():
    status = '✅ Ready' if path.exists() else '⏳ Pending download'
    print(f'{status}  {name}')

print('\nRun notebook 02 once all datasets are ready.')

📦 Data Inventory
✅ Ready  World Bank SDG-7 time series
✅ Ready  Kenya county shapefile
✅ Ready  Kenya DHS 2022 household data
✅ Ready  Facebook Relative Wealth Index

Run notebook 02 once all datasets are ready.
